# Transfer experiments

> Pretrain the CNN on OSSL, train and score zero-shot, head-only, fine-tuned, fully fine-tuned and Fukushima-only models over several seeds, and check a model within a target range

In [ ]:
#| default_exp transfer

In [ ]:
#| export
import copy
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np, pandas as pd, torch
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
import fastai.callback.schedule
from fastai.torch_core import set_seed
from soilspectfm.preprocessing import SNV, SavitzkyGolay
from soilspecdl.all import (get_spectral_dls, get_splitter, get_mask_splitter, spectral_learner, SpectralCNN, BatchSNV, cross_validate,
    train_or_load, load_weights, adabn, test_preds, regression_scores)
from ossl_fuku_transfer.data import data_path, shared_wns

In [ ]:
from fastcore.test import *

`03_kex` and `04_cec` use this module on real data. The examples here use small random arrays, so they run without the data.

## Models and preprocessing

In [ ]:
#| export
def preprocessings()->dict: # Name: unfitted scikit-learn pipeline
    "SNV, and SNV followed by a Savitzky-Golay first derivative"
    return {'snv': Pipeline([('snv', SNV())]),
            'snv_d1': Pipeline([('snv', SNV()), ('deriv', SavitzkyGolay(deriv=1))])}

Each preprocessing is fitted on OSSL and applied unchanged to Fukushima. Every model is soilspecdl's default `SpectralCNN`, the CNN of Albinet et al. (2022). In every model, `BatchSNV` then standardizes each spectrum in a batch. The CNN takes a batch of spectra and returns one prediction per spectrum:


In [ ]:
test_eq(SpectralCNN()(torch.randn(4, 1, len(shared_wns))).shape, (4,))
list(preprocessings())

## Pretraining

In [ ]:
#| export
def model_dir()->Path: # Folder of pretrained weights and training histories
    "`models/ossl-fuku-transfer` in `data_path()`"
    p = data_path()/'models'/'ossl-fuku-transfer'
    p.mkdir(parents=True, exist_ok=True)
    return p

def pretrain(
    X, # Preprocessed OSSL spectra
    y, # Targets, such as `log1p` of a soil property
    fname, # File name for the weights, without extension
    seed=42, # Seed for the validation split and the training
    epochs=20, # Number of one-cycle epochs
    lr=1e-3, # Maximum learning rate
)->SpectralCNN: # The trained model, on the CPU
    "Train a `SpectralCNN` on OSSL, or load the weights that an earlier run saved under `fname`"
    return train_or_load(model_dir()/f'{fname}.pth', X, y, shared_wns, epochs=epochs, lr=lr, seed=seed)

def load_pretrained(fname)->SpectralCNN: # The model, on the CPU
    "Load the weights that `pretrain` saved under `fname`"
    return load_weights(model_dir()/f'{fname}.pth')

def pretrain_history(fname)->pd.DataFrame: # One row per epoch: losses and validation R²
    "The training history that `pretrain` saved under `fname`"
    return pd.read_csv(model_dir()/f'{fname}.csv', index_col='epoch')

`pretrain` calls soilspecdl's `train_or_load` with the project's grid, `shared_wns`, and weights folder, `model_dir()`. It splits OSSL at random into 90% training and 10% validation, and trains for `epochs` one-cycle epochs, as in the soilspecdl quick start. It saves the weights and the training history in `model_dir()`. A later call with the same `fname` loads them instead of training again. Delete the files to train again. Training on an Apple silicon GPU (MPS) is not fully deterministic, even with a fixed seed.


In [ ]:
#| export
def range_check(
    X, # Preprocessed OSSL spectra, as passed to `pretrain`
    y, # Targets, as passed to `pretrain`
    model, # Model returned by `pretrain`
    lo, # Lower bound of the range, in the units of `y`
    hi, # Upper bound of the range
    seed=42, # Seed passed to `pretrain`
)->dict: # Sample count, R² and correlation, for all validation samples and those in range
    "Score `model` on its OSSL validation samples: all of them, and those with `y` between `lo` and `hi`"
    dls, _ = get_spectral_dls(X, y, shared_wns, splitter=get_splitter(0.9, 0.1, seed=seed), batch_tfms=[BatchSNV()])
    learn = spectral_learner(dls, copy.deepcopy(model))
    with learn.no_bar(): p, t = learn.get_preds(ds_idx=1)
    p, t = p.numpy().ravel(), t.numpy().ravel()
    m = (t >= lo) & (t <= hi)
    return {'n all': len(t), 'R² all': r2_score(t, p), 'r all': np.corrcoef(t, p)[0, 1],
            'n in range': int(m.sum()), 'R² in range': r2_score(t[m], p[m]), 'r in range': np.corrcoef(t[m], p[m])[0, 1]}

A model scored over a narrow range has a lower R² than over a wide one, because the targets vary less. `range_check` measures a pretrained model on the part of its OSSL validation set that covers the Fukushima range. This gives the reference for Fukushima results.

## Fukushima models

In [ ]:
#| export
def fuku_dls(
    X, # Preprocessed Fukushima spectra
    y, # Targets
    is_test, # True for the samples of the fixed test set
    seed=0, # Seed for choosing the validation samples
    valid_pct=0.1, # Share of the training samples used for validation
):
    "`DataLoaders` over all Fukushima spectra, with the test set indices in `test_idx`"
    return get_spectral_dls(X, y, shared_wns, splitter=get_mask_splitter(is_test, valid_pct, seed), batch_tfms=[BatchSNV()])[0]

With 100 samples of which 20 are test samples, 8 of the other 80 go to validation:

In [ ]:
rng = np.random.default_rng(0)
X_syn, y_syn = rng.random((100, len(shared_wns))), rng.random(100)
is_test_syn = np.arange(100) >= 80
dls_syn = fuku_dls(X_syn, y_syn, is_test_syn)
test_eq((len(dls_syn.train_ds), len(dls_syn.valid_ds), len(dls_syn.test_idx)), (72, 8, 20))

In [ ]:
#| export
model_names = ('zero-shot', 'zero-shot AdaBN', 'head only', 'fine-tuned', 'fully fine-tuned', 'Fukushima only') # Models that `fit_models` trains

def fit_models(
    pretrained, # Model returned by `pretrain`
    X, # Preprocessed Fukushima spectra
    y, # Targets
    is_test, # True for the samples of the fixed test set
    seed=0, # Seed for the validation split and the training
    epochs=20, # Training epochs after the first, frozen, epoch of fine-tuning
    lr=1e-3, # Maximum learning rate
    schedules=None, # Model name: (epochs, maximum learning rate), for 'fully fine-tuned' and 'Fukushima only'
)->dict: # Model name: test predictions
    "Test predictions of the models in `model_names`"
    schedules = schedules or {}
    dls, res = fuku_dls(X, y, is_test, seed), {}
    def _fit(learn, fit):
        with learn.no_bar(), learn.no_logging(): fit(learn)
        return test_preds(learn)
    res['zero-shot'] = test_preds(spectral_learner(dls, copy.deepcopy(pretrained)))
    res['zero-shot AdaBN'] = test_preds(spectral_learner(dls, adabn(pretrained, dls)))
    set_seed(seed)
    learn = spectral_learner(dls, copy.deepcopy(pretrained), train_bn=False)
    learn.freeze()
    res['head only'] = _fit(learn, lambda l: l.fit_one_cycle(epochs + 1, lr_max=lr))
    set_seed(seed)
    res['fine-tuned'] = _fit(spectral_learner(dls, copy.deepcopy(pretrained)), lambda l: l.fine_tune(epochs, base_lr=lr))
    set_seed(seed)
    ep, lr_ = schedules.get('fully fine-tuned', (epochs + 1, lr))
    res['fully fine-tuned'] = _fit(spectral_learner(dls, copy.deepcopy(pretrained)), lambda l: l.fit_one_cycle(ep, lr_max=lr_))
    set_seed(seed)
    ep, lr_ = schedules.get('Fukushima only', (epochs + 1, lr))
    res['Fukushima only'] = _fit(spectral_learner(dls, SpectralCNN()), lambda l: l.fit_one_cycle(ep, lr_max=lr_))
    return res

The six models see the same validation split. The trained models see the same number of epochs:

- Zero-shot predicts with the pretrained model as it is.
- Zero-shot AdaBN predicts with the pretrained model after `adabn` has recomputed its BatchNorm statistics on the non-test Fukushima spectra. It uses no Fukushima targets.
- Head only trains the pretrained head, a linear layer of 257 weights, for `epochs + 1` epochs. The backbone weights stay fixed, including the BatchNorm scale and shift. The BatchNorm running statistics still update to the Fukushima batches during training.
- Fine-tuned uses fastai's `fine_tune`: one epoch with the backbone frozen at learning rate `lr`, then `epochs` epochs of the whole model. In that second phase the head's maximum learning rate is `lr / 2`, and the backbone's is `lr / 200`. With `lr=1e-3`, the backbone trains at most at 5e-6, 200 times lower than the Fukushima-only model, and stays close to its pretrained weights.
- Fully fine-tuned trains the whole pretrained model with the schedule of the Fukushima-only model: `epochs + 1` one-cycle epochs at maximum learning rate `lr`. It differs from the Fukushima-only model only in its starting weights.
- Fukushima only trains a `SpectralCNN` from random weights for `epochs + 1` epochs.

`schedules` gives the fully fine-tuned and Fukushima-only models their own number of one-cycle epochs and maximum learning rate, such as ones chosen with `cv_curves` and `cv_final` below. Without it, both use the defaults above.

With an untrained model in place of a pretrained one and one epoch, each model predicts the 20 test samples:

In [ ]:
preds_syn = fit_models(SpectralCNN(), X_syn, y_syn, is_test_syn, epochs=1)
test_eq({k: v.shape for k, v in preds_syn.items()}, {k: (20,) for k in model_names})

## Scores

In [ ]:
#| export
def scores(
    y, # Observed targets, as `log1p` of the property
    p, # Predicted targets
)->dict: # R², correlation and concordance of the targets, RMSE and bias of the property
    "`regression_scores` of `log1p` targets, with the RMSE and bias of the property itself"
    return regression_scores(y, p, inv=np.expm1)

A prediction that adds a constant to every `log1p` target keeps a correlation of 1, but loses R², and gains a bias:

In [ ]:
s = scores(y_syn, y_syn + 0.2)
test_close(s['r'], 1)
assert s['R²'] < 1 and s['bias'] > 0
{k: round(v, 3) for k, v in s.items()}

In [ ]:
#| export
def compare(
    pretrained, # Preprocessing name: model returned by `pretrain`
    X, # Preprocessing name: preprocessed Fukushima spectra
    y, # Targets
    is_test, # True for the samples of the fixed test set
    seeds=range(5), # Seeds to repeat `fit_models` with
    **kwargs, # Passed to `fit_models`
)->tuple: # (scores, one row per preprocessing, model and seed; predictions by (preprocessing, model, seed))
    "Run `fit_models` for each preprocessing and seed, and score the test predictions"
    rows, preds = [], {}
    for name, model in pretrained.items():
        for seed in seeds:
            for m, p in fit_models(model, X[name], y, is_test, seed, **kwargs).items():
                rows.append({'preprocessing': name, 'model': m, 'seed': seed, **scores(y[is_test], p)})
                preds[name, m, seed] = p
    return pd.DataFrame(rows), preds

def summarise(res:pd.DataFrame)->pd.DataFrame: # Mean and standard deviation over seeds
    "Mean and standard deviation of each score over the seeds, by preprocessing and model"
    out = res.groupby(['preprocessing', 'model'])[['R²', 'r', 'RMSE', 'bias']].agg(['mean', 'std'])
    return out.reindex(list(model_names), level='model')

`compare` repeats the Fukushima-side training over several seeds. The seed changes the validation split and the initial weights of the new heads and models. The pretrained models stay the same. `summarise` gives the mean and standard deviation of each score:

In [ ]:
res_syn, _ = compare({'snv': SpectralCNN()}, {'snv': X_syn}, y_syn, is_test_syn, seeds=range(2), epochs=1)
test_eq(len(res_syn), 2 * len(model_names))
summarise(res_syn).shape

## Plots and error analysis

In [ ]:
#| export
def plot_preds(
    preds, # Predictions returned by `compare`
    y, # Observed test targets
    seed=0, # Seed whose predictions to plot
    label='log1p(target)', # Axis label
):
    "Predicted against observed test targets, one panel per preprocessing and model"
    names = list(dict.fromkeys(k[0] for k in preds))
    models = list(dict.fromkeys(k[1] for k in preds))
    fig, axs = plt.subplots(len(names), len(models), figsize=(3 * len(models), 3 * len(names)), sharex=True, sharey=True, squeeze=False)
    lims = [y.min(), y.max()]
    for row, n in zip(axs, names):
        for ax, m in zip(row, models):
            p = preds[n, m, seed]
            ax.scatter(y, p, s=5, alpha=0.5)
            ax.plot(lims, lims, c='k', lw=0.8)
            ax.set_title(f'{n}, {m}: R² {r2_score(y, p):.2f}', fontsize=9)
    for ax in axs[-1]: ax.set_xlabel(f'Observed {label}')
    for ax in axs[:, 0]: ax.set_ylabel(f'Predicted {label}')
    return fig

In [ ]:
plot_preds(compare({'snv': SpectralCNN()}, {'snv': X_syn}, y_syn, is_test_syn, seeds=range(1), epochs=1)[1], y_syn[is_test_syn]);

In [ ]:
#| export
def error_table(
    preds, # Predictions returned by `compare`
    y, # Observed test targets
    labels, # Group label of each test sample, such as its soil type
    name, # Preprocessing to report
    models=model_names[2:], # Models to report
)->pd.DataFrame: # One row per group: mean absolute error of each model, and sample count
    "Mean absolute error of the seed-averaged predictions of each model, by group"
    seeds = sorted({k[2] for k in preds if k[0] == name})
    err = {m: np.abs(np.mean([preds[name, m, s] for s in seeds], axis=0) - y) for m in models}
    return pd.DataFrame(err).assign(n=1).groupby(np.asarray(labels)).agg({**{m: 'mean' for m in models}, 'n': 'sum'})

`error_table` averages each model's predictions over the seeds, then reports the mean absolute error of each group:

In [ ]:
tab = error_table(compare({'snv': SpectralCNN()}, {'snv': X_syn}, y_syn, is_test_syn, seeds=range(2), epochs=1)[1], y_syn[is_test_syn], np.repeat(['a', 'b'], 10), 'snv')
test_eq(tab.n.tolist(), [10, 10])
tab.round(3)

## Choosing a schedule

The schedules of the fully fine-tuned and Fukushima-only models are chosen on the Fukushima training samples only. The test set plays no part in the choice. `fuku_lr_find` runs fastai's learning rate finder from the pretrained weights, or from random weights when `model` is `None`:

In [ ]:
#| export
def fuku_lr_find(
    model, # Model returned by `pretrain`, or None for random weights
    X, # Preprocessed Fukushima spectra
    y, # Targets
    is_test, # True for the samples of the fixed test set
    seed=0, # Seed for the validation split
    show_plot=True, # Plot the loss against the learning rate
):
    "Run `lr_find` on the Fukushima training samples"
    learn = spectral_learner(fuku_dls(X, y, is_test, seed), SpectralCNN() if model is None else copy.deepcopy(model))
    with learn.no_bar(): return learn.lr_find(show_plot=show_plot)

In [ ]:
fuku_lr_find(None, X_syn, y_syn, is_test_syn, show_plot=False)

In [ ]:
#| export
def cv_curves(
    model, # Model returned by `pretrain`, or None for random weights
    X, # Preprocessed Fukushima spectra
    y, # Targets
    is_test, # True for the samples of the fixed test set
    lr, # Maximum learning rate
    epochs, # Number of one-cycle epochs
    fname=None, # File name to cache the results under, without extension
    n_splits=5, # Number of folds
    seed=42, # Seed for the folds
)->pd.DataFrame: # One row per fold and epoch, from `cross_validate`
    "Cross-validate one-cycle training on the Fukushima training samples"
    fn = model_dir()/f'{fname}.csv' if fname else None
    if fn and fn.exists(): return pd.read_csv(fn)
    model_fn = SpectralCNN if model is None else (lambda: copy.deepcopy(model))
    def fit(learn):
        with learn.no_bar(), learn.no_logging(): learn.fit_one_cycle(epochs, lr_max=lr)
    res = cross_validate(X[~is_test], y[~is_test], shared_wns, model_fn, fit, n_splits=n_splits, seed=seed, batch_tfms=[BatchSNV()])
    if fn: res.to_csv(fn, index=False)
    return res

def cv_final(res:pd.DataFrame)->dict: # Mean and standard deviation over folds
    "Validation loss and R² at the last epoch of `cv_curves`, as mean and standard deviation over folds"
    last = res[res.epoch == res.epoch.max()]
    return {'valid_loss': last.valid_loss.mean(), 'valid_loss sd': last.valid_loss.std(),
            'R²': last.r2_score.mean(), 'R² sd': last.r2_score.std()}

`cv_curves` trains one model per fold on the training samples, and records the losses and R² after every epoch. `plot_cv` from soilspecdl plots them. Results are cached under `fname` in `model_dir()`, so a notebook rerun does not repeat the training. `cv_final` summarises the last epoch, which is the one a one-cycle schedule ends on. On random data, R² stays near 0 or below:

In [ ]:
res_cv = cv_curves(None, X_syn, y_syn, is_test_syn, lr=1e-3, epochs=2, n_splits=2)
test_eq(len(res_cv), 2 * 2)
{k: round(v, 3) for k, v in cv_final(res_cv).items()}

## Joint training

Fine-tuning trains on OSSL first and on Fukushima afterwards. Joint training uses both at once: one model trains from random weights on all OSSL spectra and the Fukushima training samples together. The Fukushima samples are repeated so that they make up about `share` of the training samples. Without repetition, they would be under 2% of them. Validation and test use Fukushima samples only, as in `fuku_dls`:

In [ ]:
#| export
def joint_dls(
    Xo, # Preprocessed OSSL spectra
    yo, # OSSL targets
    Xf, # Preprocessed Fukushima spectra
    yf, # Fukushima targets
    is_test, # True for the Fukushima samples of the fixed test set
    share=0.25, # Target share of Fukushima samples in the training set
    seed=0, # Seed for choosing the Fukushima validation samples
    valid_pct=0.1, # Share of the Fukushima training samples used for validation
):
    "`DataLoaders` training on OSSL and Fukushima together, validating and testing on Fukushima"
    n_o, f_idx = len(Xo), np.flatnonzero(~is_test)
    np.random.RandomState(seed).shuffle(f_idx)
    n_valid = int(len(f_idx) * valid_pct)
    f_valid, f_train = f_idx[:n_valid] + n_o, f_idx[n_valid:] + n_o
    reps = max(1, round(share * n_o / ((1 - share) * len(f_train))))
    train_idx = np.concatenate([np.arange(n_o), np.tile(f_train, reps)])
    test_idx = np.flatnonzero(is_test) + n_o
    split = lambda X, y: (train_idx, f_valid, test_idx)
    dls = get_spectral_dls(np.concatenate([Xo, Xf]), np.concatenate([yo, yf]), shared_wns, splitter=split, batch_tfms=[BatchSNV()])[0]
    dls.test_idx, dls.reps = test_idx, reps
    return dls

With 200 OSSL samples and 72 Fukushima training samples, a share of 0.5 repeats the Fukushima samples 3 times:

In [ ]:
Xo_syn, yo_syn = rng.random((200, len(shared_wns))), rng.random(200)
dls_j = joint_dls(Xo_syn, yo_syn, X_syn, y_syn, is_test_syn, share=0.5)
test_eq((dls_j.reps, len(dls_j.train_ds), len(dls_j.valid_ds), len(dls_j.test_idx)), (3, 200 + 3 * 72, 8, 20))

In [ ]:
#| export
def fit_joint(
    Xo, yo, Xf, yf, is_test, # As in `joint_dls`
    share=0.25, # Target share of Fukushima samples in the training set
    seed=0, # Seed for the validation split and the training
    epochs=20, # Number of one-cycle epochs
    lr=1e-3, # Maximum learning rate
    fname=None, # File name to cache the test predictions under, without extension
)->np.ndarray: # One prediction per Fukushima test sample
    "Train `SpectralCNN` from random weights on OSSL and Fukushima together, and predict the Fukushima test set"
    fn = model_dir()/f'{fname}.npy' if fname else None
    if fn and fn.exists(): return np.load(fn)
    set_seed(seed)
    learn = spectral_learner(joint_dls(Xo, yo, Xf, yf, is_test, share, seed), SpectralCNN())
    with learn.no_bar(), learn.no_logging(): learn.fit_one_cycle(epochs, lr_max=lr)
    p = test_preds(learn)
    if fn: np.save(fn, p)
    return p

In [ ]:
test_eq(fit_joint(Xo_syn, yo_syn, X_syn, y_syn, is_test_syn, share=0.5, epochs=1).shape, (20,))

## Learning curve

`subset_preds` trains a CNN on `n` Fukushima training samples drawn at random with `seed`, and predicts the fixed test set. It starts from the weights of `model`, or from random weights if `model` is None. `model` can also be a function that takes the indices of the drawn samples and returns a model, for starting points that depend on the training samples. The function is called after the seed is set. As in `fuku_dls`, a tenth of the `n` samples is kept for validation.

In [ ]:
#| export
def subset_preds(
    model, # Starting model, such as one returned by `pretrain`; random weights if None; or a function of the training indices that returns one
    X, # Preprocessed Fukushima spectra
    y, # Targets
    is_test, # True for the samples of the fixed test set
    n, # Number of training samples to draw
    seed=0, # Seed for drawing the samples, the validation split and the training
    epochs=20, # Number of one-cycle epochs
    lr=1e-3, # Maximum learning rate
)->np.ndarray: # One prediction per test sample
    "Test predictions of a CNN trained on `n` random training samples"
    sub = np.random.RandomState(seed).choice(np.flatnonzero(~is_test), n, replace=False)
    idx = np.concatenate([sub, np.flatnonzero(is_test)])
    dls = fuku_dls(X[idx], y[idx], is_test[idx], seed)
    set_seed(seed)
    if model is None: m = SpectralCNN()
    elif isinstance(model, torch.nn.Module): m = copy.deepcopy(model)
    else: m = model(sub)
    learn = spectral_learner(dls, m)
    with learn.no_bar(), learn.no_logging(): learn.fit_one_cycle(epochs, lr_max=lr)
    return test_preds(learn)

In [ ]:
p_sub = subset_preds(None, X_syn, y_syn, is_test_syn, 40, epochs=1)
test_eq(p_sub.shape, (20,))
seen = []
p_fn = subset_preds(lambda idx: seen.append(idx) or SpectralCNN(), X_syn, y_syn, is_test_syn, 40, epochs=1)
test_eq((len(seen[0]), is_test_syn[seen[0]].any()), (40, False))